In [0]:
%run ./00_config

✅ Config loaded : billing.sales_data
📅 Target date   : 2026-10-03


In [0]:
# ============================================================
# GOLD STORE PERFORMANCE
# Production-grade / Serverless-safe
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

import uuid
import time


# ============================================================
# CONFIGURATION
# ============================================================

SOURCE_TABLE = f"{catalog_name}.{schema_name}.billing_silver"
TARGET_TABLE = f"{catalog_name}.{schema_name}.gold_store_performance"

# TARGET_DATE is expected to already be defined
# by the parent notebook / Databricks Job parameter.

EXCLUDED_STORES = [
    "ho marlboro"
]

PIPELINE_RUN_ID = str(uuid.uuid4())

START_TIME = time.time()


print("=" * 80)
print("GOLD STORE PERFORMANCE")
print("=" * 80)
print(f"Source table    : {SOURCE_TABLE}")
print(f"Target table    : {TARGET_TABLE}")
print(f"Target date     : {TARGET_DATE}")
print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
print(f"Excluded stores : {EXCLUDED_STORES}")
print("=" * 80)


# ============================================================
# HELPERS
# ============================================================

def fail(message):
    raise RuntimeError(f"[GOLD_STORE] {message}")


def assert_columns_exist(df, required_columns, df_name):
    missing = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing:
        fail(
            f"{df_name} is missing required columns: {missing}"
        )


def schema_map(df):
    return {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }


# ============================================================
# 1. READ SILVER
# ============================================================

print("\n[1/10] Reading Silver table...")

try:
    silver = spark.table(SOURCE_TABLE)
except AnalysisException as e:
    fail(
        f"Unable to read source table {SOURCE_TABLE}: {str(e)}"
    )


required_source_columns = [
    "source_date",
    "storeName",
    "orderDate",
    "invoice",
    "totalProductPrice"
]

assert_columns_exist(
    silver,
    required_source_columns,
    "billing_silver"
)

print("Silver table loaded successfully.")


# ============================================================
# 2. FILTER TARGET SOURCE DATE
# ============================================================

print("\n[2/10] Filtering Silver for target source date...")

silver_target = (
    silver
    .filter(
        F.col("source_date").cast("string") == str(TARGET_DATE)
    )
)

silver_target_count = silver_target.count()

print(
    f"Silver rows for source_date={TARGET_DATE}: "
    f"{silver_target_count:,}"
)

if silver_target_count == 0:
    fail(
        f"No Silver records found for TARGET_DATE={TARGET_DATE}"
    )


# ============================================================
# 3. NORMALIZE INPUT COLUMNS
# ============================================================

print("\n[3/10] Normalizing Store Performance input...")

silver_target = (
    silver_target

    .withColumn(
        "_store_name",
        F.trim(
            F.col("storeName").cast("string")
        )
    )

    .withColumn(
        "_order_date",
        F.to_date(
            F.col("orderDate")
        )
    )

    .withColumn(
        "_invoice",
        F.trim(
            F.col("invoice").cast("string")
        )
    )

    .withColumn(
        "_sales",
        F.col("totalProductPrice").cast("double")
    )
)


# ============================================================
# 4. EXCLUDE STORES
# ============================================================

print("\n[4/10] Applying store exclusions...")

excluded_condition = (
    F.lower(
        F.trim(
            F.col("_store_name")
        )
    ).isin(EXCLUDED_STORES)
)

excluded_count = (
    silver_target
    .filter(excluded_condition)
    .count()
)

print(
    f"Rows excluded: {excluded_count:,}"
)

silver_filtered = (
    silver_target
    .filter(~excluded_condition)
)


# ============================================================
# 5. SOURCE DATA QUALITY VALIDATION
# ============================================================

print("\n[5/10] Running source data quality checks...")


# ------------------------------------------------------------
# Store name
# ------------------------------------------------------------

null_store_count = (
    silver_filtered
    .filter(
        F.col("_store_name").isNull()
        |
        (F.col("_store_name") == "")
    )
    .count()
)

if null_store_count > 0:
    fail(
        f"Silver contains {null_store_count:,} "
        f"rows with NULL/blank storeName"
    )


# ------------------------------------------------------------
# Order date
# ------------------------------------------------------------

null_order_date_count = (
    silver_filtered
    .filter(
        F.col("_order_date").isNull()
    )
    .count()
)

if null_order_date_count > 0:
    fail(
        f"Silver contains {null_order_date_count:,} "
        f"rows with NULL orderDate"
    )


# ------------------------------------------------------------
# Invoice
# ------------------------------------------------------------

null_invoice_count = (
    silver_filtered
    .filter(
        F.col("_invoice").isNull()
        |
        (F.col("_invoice") == "")
    )
    .count()
)

if null_invoice_count > 0:
    fail(
        f"Silver contains {null_invoice_count:,} "
        f"rows with NULL/blank invoice"
    )


# ------------------------------------------------------------
# Sales
# ------------------------------------------------------------

null_sales_count = (
    silver_filtered
    .filter(
        F.col("_sales").isNull()
    )
    .count()
)

if null_sales_count > 0:
    fail(
        f"Silver contains {null_sales_count:,} "
        f"rows with NULL totalProductPrice"
    )


filtered_count = silver_filtered.count()

if filtered_count == 0:
    fail(
        "No records remain after store exclusions."
    )

print(
    f"Rows after exclusions: {filtered_count:,}"
)


# ============================================================
# 6. SILVER CONTROL TOTAL
# ============================================================

print("\n[6/10] Calculating Silver control totals...")

silver_total_sales = (
    silver_filtered
    .agg(
        F.sum("_sales").alias("total_sales")
    )
    .collect()[0]["total_sales"]
)

if silver_total_sales is None:
    silver_total_sales = 0.0

silver_total_sales = float(
    silver_total_sales
)

print(
    f"Silver totalProductPrice: "
    f"{silver_total_sales:,.2f}"
)


# ============================================================
# 7. BUILD GOLD STORE PERFORMANCE
# ============================================================

print("\n[7/10] Building Gold Store Performance...")

store_df = (
    silver_filtered

    .groupBy(
        F.col("_store_name").alias("storeName"),
        F.col("_order_date").alias("orderDate")
    )

    .agg(
        F.countDistinct("_invoice")
        .cast("long")
        .alias("nooforder"),

        F.round(
            F.sum("_sales"),
            2
        )
        .cast("double")
        .alias("sales")
    )

    .withColumn(
        "aov",
        F.when(
            F.col("nooforder") > 0,
            F.round(
                F.col("sales") /
                F.col("nooforder"),
                2
            )
        )
        .otherwise(
            F.lit(0.0)
        )
        .cast("double")
    )

    # IMPORTANT:
    # source_date EXISTS in the production Gold table.
    # It is a STRING and must be populated explicitly.
    .withColumn(
        "source_date",
        F.lit(str(TARGET_DATE)).cast("string")
    )

    .select(
        "storeName",
        "orderDate",
        "nooforder",
        "sales",
        "aov",
        "source_date"
    )
)


# ============================================================
# 8. VALIDATE GENERATED GOLD DATA
# ============================================================

print("\n[8/10] Validating generated Gold data...")

generated_count = store_df.count()

if generated_count == 0:
    fail(
        "Generated Gold Store Performance contains 0 rows."
    )

print(
    f"Generated Gold rows: {generated_count:,}"
)


# ------------------------------------------------------------
# Expected production schema
# ------------------------------------------------------------

expected_schema = {
    "storeName": "string",
    "orderDate": "date",
    "nooforder": "bigint",
    "sales": "double",
    "aov": "double",
    "source_date": "string"
}

actual_schema = schema_map(store_df)

if actual_schema != expected_schema:
    fail(
        "Generated Gold schema mismatch.\n"
        f"Expected: {expected_schema}\n"
        f"Actual  : {actual_schema}"
    )

print(
    "Generated schema validation passed."
)


# ------------------------------------------------------------
# Null grain
# ------------------------------------------------------------

null_grain_count = (
    store_df
    .filter(
        F.col("storeName").isNull()
        |
        F.col("orderDate").isNull()
    )
    .count()
)

if null_grain_count > 0:
    fail(
        f"Gold contains {null_grain_count:,} "
        f"rows with NULL grain keys."
    )


# ------------------------------------------------------------
# Source date validation
# ------------------------------------------------------------

invalid_source_date_count = (
    store_df
    .filter(
        F.col("source_date") != str(TARGET_DATE)
    )
    .count()
)

if invalid_source_date_count > 0:
    fail(
        f"Gold contains {invalid_source_date_count:,} "
        f"rows with incorrect source_date."
    )


# ------------------------------------------------------------
# Order count validation
# ------------------------------------------------------------

invalid_order_count = (
    store_df
    .filter(
        F.col("nooforder").isNull()
        |
        (F.col("nooforder") <= 0)
    )
    .count()
)

if invalid_order_count > 0:
    fail(
        f"Gold contains {invalid_order_count:,} "
        f"rows with invalid nooforder."
    )


# ------------------------------------------------------------
# Duplicate grain
# ------------------------------------------------------------

duplicate_grain_count = (
    store_df
    .groupBy(
        "storeName",
        "orderDate"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

if duplicate_grain_count > 0:
    fail(
        f"Gold contains {duplicate_grain_count:,} "
        f"duplicate (storeName, orderDate) groups."
    )

print(
    "Duplicate grain validation passed."
)


# ------------------------------------------------------------
# Excluded stores
# ------------------------------------------------------------

excluded_gold_count = (
    store_df
    .filter(
        F.lower(
            F.trim(
                F.col("storeName")
            )
        ).isin(EXCLUDED_STORES)
    )
    .count()
)

if excluded_gold_count > 0:
    fail(
        f"Gold contains {excluded_gold_count:,} "
        f"rows belonging to excluded stores."
    )


# ------------------------------------------------------------
# Gold sales
# ------------------------------------------------------------

gold_total_sales = (
    store_df
    .agg(
        F.sum("sales").alias("total_sales")
    )
    .collect()[0]["total_sales"]
)

if gold_total_sales is None:
    gold_total_sales = 0.0

gold_total_sales = float(
    gold_total_sales
)

sales_difference = abs(
    silver_total_sales -
    gold_total_sales
)

print(
    f"Silver sales : {silver_total_sales:,.2f}"
)

print(
    f"Gold sales   : {gold_total_sales:,.2f}"
)

print(
    f"Difference   : {sales_difference:,.2f}"
)


# Sales are rounded to 2 decimal places at the
# store/orderDate grain.
sales_tolerance = max(
    0.01,
    generated_count * 0.01
)

if sales_difference > sales_tolerance:
    fail(
        "Silver → Gold sales reconciliation failed.\n"
        f"Silver sales : {silver_total_sales:,.2f}\n"
        f"Gold sales   : {gold_total_sales:,.2f}\n"
        f"Difference   : {sales_difference:,.2f}\n"
        f"Tolerance    : {sales_tolerance:,.2f}"
    )

print(
    "Silver → Gold sales reconciliation passed."
)


# ============================================================
# 9. VALIDATE EXISTING TARGET SCHEMA
# ============================================================

print("\n[9/10] Validating existing target table...")

target_exists = spark.catalog.tableExists(
    TARGET_TABLE
)

if target_exists:

    target = spark.table(
        TARGET_TABLE
    )

    expected_target_schema = {
        "storeName": "string",
        "orderDate": "date",
        "nooforder": "bigint",
        "sales": "double",
        "aov": "double",
        "source_date": "string"
    }

    actual_target_schema = schema_map(
        target
    )

    if actual_target_schema != expected_target_schema:
        fail(
            "Existing target schema does not match "
            "the established production schema.\n"
            f"Expected: {expected_target_schema}\n"
            f"Actual  : {actual_target_schema}"
        )

    print(
        "Existing target schema validation passed."
    )

else:

    print(
        "Target table does not exist. "
        "It will be created with the established schema."
    )


# ============================================================
# 10. ATOMIC WRITE
# ============================================================

print("\n[10/10] Writing Gold Store Performance...")

if target_exists:

    print(
        f"Atomic replacement for "
        f"orderDate={TARGET_DATE}"
    )

    (
        store_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"orderDate = '{TARGET_DATE}'"
        )
        .saveAsTable(TARGET_TABLE)
    )

    print(
        "Atomic replaceWhere write committed."
    )

else:

    (
        store_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

    print(
        "Gold Store Performance table created."
    )


# ============================================================
# POST-WRITE VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("POST-WRITE VALIDATION")
print("=" * 80)


target_after = spark.table(
    TARGET_TABLE
)


# ------------------------------------------------------------
# Schema validation
# ------------------------------------------------------------

post_schema = schema_map(
    target_after
)

if post_schema != expected_schema:
    fail(
        "Post-write target schema validation failed.\n"
        f"Expected: {expected_schema}\n"
        f"Actual  : {post_schema}"
    )

print(
    "✓ Target schema correct."
)


# ------------------------------------------------------------
# Target date rows
# ------------------------------------------------------------

post_target = (
    target_after
    .filter(
        F.col("orderDate") ==
        F.to_date(
            F.lit(TARGET_DATE)
        )
    )
)

post_count = post_target.count()

print(
    f"Expected rows for {TARGET_DATE}: "
    f"{generated_count:,}"
)

print(
    f"Actual rows for {TARGET_DATE}:   "
    f"{post_count:,}"
)

if post_count != generated_count:
    fail(
        "Post-write row count mismatch.\n"
        f"Expected: {generated_count:,}\n"
        f"Actual:   {post_count:,}"
    )


# ------------------------------------------------------------
# Post-write source_date
# ------------------------------------------------------------

post_invalid_source_date = (
    post_target
    .filter(
        F.col("source_date") !=
        str(TARGET_DATE)
    )
    .count()
)

if post_invalid_source_date > 0:
    fail(
        f"Post-write target contains "
        f"{post_invalid_source_date:,} rows "
        f"with incorrect source_date."
    )


# ------------------------------------------------------------
# Post-write sales
# ------------------------------------------------------------

post_sales = (
    post_target
    .agg(
        F.sum("sales").alias("total_sales")
    )
    .collect()[0]["total_sales"]
)

if post_sales is None:
    post_sales = 0.0

post_sales = float(
    post_sales
)

post_sales_difference = abs(
    gold_total_sales -
    post_sales
)

print(
    f"Expected sales: {gold_total_sales:,.2f}"
)

print(
    f"Actual sales:   {post_sales:,.2f}"
)

print(
    f"Difference:     {post_sales_difference:,.2f}"
)

if post_sales_difference > 0.01:
    fail(
        "Post-write sales validation failed."
    )


# ------------------------------------------------------------
# Post-write NULL grain
# ------------------------------------------------------------

post_null_grain = (
    post_target
    .filter(
        F.col("storeName").isNull()
        |
        F.col("orderDate").isNull()
    )
    .count()
)

if post_null_grain > 0:
    fail(
        f"Post-write target contains "
        f"{post_null_grain:,} NULL grain rows."
    )


# ------------------------------------------------------------
# Post-write duplicate grain
# ------------------------------------------------------------

post_duplicate_grain = (
    post_target
    .groupBy(
        "storeName",
        "orderDate"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

if post_duplicate_grain > 0:
    fail(
        f"Post-write target contains "
        f"{post_duplicate_grain:,} duplicate "
        f"(storeName, orderDate) groups."
    )


# ------------------------------------------------------------
# Post-write excluded stores
# ------------------------------------------------------------

post_excluded_count = (
    post_target
    .filter(
        F.lower(
            F.trim(
                F.col("storeName")
            )
        ).isin(EXCLUDED_STORES)
    )
    .count()
)

if post_excluded_count > 0:
    fail(
        f"Post-write target contains "
        f"{post_excluded_count:,} excluded-store rows."
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

duration = time.time() - START_TIME

print("\n" + "=" * 80)
print("GOLD STORE PERFORMANCE — SUCCESS")
print("=" * 80)

print(f"Pipeline Run ID      : {PIPELINE_RUN_ID}")
print(f"Target Date          : {TARGET_DATE}")
print(f"Silver Input Rows    : {silver_target_count:,}")
print(f"Rows After Exclusion : {filtered_count:,}")
print(f"Gold Output Rows     : {generated_count:,}")
print(f"Silver Sales         : {silver_total_sales:,.2f}")
print(f"Gold Sales           : {gold_total_sales:,.2f}")
print(f"Post-write Rows      : {post_count:,}")
print(f"Post-write Sales     : {post_sales:,.2f}")
print(f"Duplicate Grain      : {post_duplicate_grain}")
print(f"Excluded Store Rows  : {post_excluded_count}")
print(f"Duration             : {duration:.2f} seconds")

print("=" * 80)
print(
    "STORE PERFORMANCE PIPELINE "
    "COMPLETED SUCCESSFULLY"
)
print("=" * 80)


# ============================================================
# DISPLAY
# ============================================================

display(
    target_after
    .filter(
        F.col("orderDate") ==
        F.to_date(
            F.lit(TARGET_DATE)
        )
    )
    .orderBy("storeName")
)

GOLD STORE PERFORMANCE
Source table    : billing.sales_data.billing_silver
Target table    : billing.sales_data.gold_store_performance
Target date     : 2026-10-03
Pipeline run ID : d58584dc-5630-4a06-a871-1c7359769a90
Excluded stores : ['ho marlboro']

[1/10] Reading Silver table...
Silver table loaded successfully.

[2/10] Filtering Silver for target source date...
Silver rows for source_date=2026-10-03: 52,995

[3/10] Normalizing Store Performance input...

[4/10] Applying store exclusions...
Rows excluded: 0

[5/10] Running source data quality checks...
Rows after exclusions: 52,995

[6/10] Calculating Silver control totals...
Silver totalProductPrice: 4,496,998.87

[7/10] Building Gold Store Performance...

[8/10] Validating generated Gold data...
Generated Gold rows: 107
Generated schema validation passed.
Duplicate grain validation passed.
Silver sales : 4,496,998.87
Gold sales   : 4,496,998.87
Difference   : 0.00
Silver → Gold sales reconciliation passed.

[9/10] Validating exi

storeName,orderDate,nooforder,sales,aov,source_date
Adchini Aurobindo Marg,2026-10-03,168,26448.25,157.43,2026-10-03
Ahmedabad Chandkheda,2026-10-03,62,11505.0,185.56,2026-10-03
Ahmedabad Gota,2026-10-03,141,27066.0,191.96,2026-10-03
Ahmedabad Green City,2026-10-03,633,104056.0,164.39,2026-10-03
Ahmedabad SKY City,2026-10-03,272,54031.0,198.64,2026-10-03
Aligarh IT Palmcourt,2026-10-03,476,66630.09,139.98,2026-10-03
Ambala NDI Plaza,2026-10-03,224,56449.01,252.0,2026-10-03
Assam Pirbi Market,2026-10-03,105,47390.0,451.33,2026-10-03
Bangaluru Rajaji Nagar,2026-10-03,14,1288.0,92.0,2026-10-03
Bareilly Baheri,2026-10-03,249,47895.0,192.35,2026-10-03
